## save combined / intermediate outputs first

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import logging
from datetime import datetime
from collections import defaultdict
import seaborn as sns

# Set up logging for better readability and debugging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

import matplotlib.pyplot as plt

# Get current date in YYYY-MM-DD format
current_date = datetime.today().strftime('%Y-%m-%d')
print(current_date)
import anndata as ad

In [ ]:
save_dir = '/rds/general/project/rleetmaa/live/kamal_hetvi_shared_data/VIEW_terekhova_synapse_10xrna_pbmc/geoguessPipelineOuts_mar25/'
batch_lst = os.listdir(save_dir)

In [ ]:
len(batch_lst)

In [ ]:
from pathlib import Path

In [ ]:
files_lst = {}

min_tot_site = 500

for batch_nm in batch_lst:
    
    parent_dir = Path(save_dir) / Path(batch_nm)

    fl_nm = f'terekhova10xPBMC_{batch_nm}_with_prev_minTotSites{min_tot_site}.h5ad'

    if fl_nm in os.listdir(parent_dir):
        files_lst[batch_nm] = parent_dir / fl_nm
    else:
        print(f'{batch_nm} does not have fl')

In [ ]:
len(files_lst) == len(batch_lst)

In [ ]:
h5_objs = []

for batch_nm in batch_lst:    

    tk_h5 = sc.read_h5ad(files_lst[batch_nm])
    del(tk_h5.uns)
    
    h5_objs.append(tk_h5)

In [ ]:
for h5o in h5_objs:
    h5o.obs = h5o.obs.set_index('cell_barcode')

In [ ]:
combined_all = ad.concat(
    h5_objs,
    join="outer",           # keep all genes; "inner" keeps only genes shared by all files
    merge="same",           # keep var columns that are identical across files
)

In [ ]:
print(combined_all.obs['given_cell_type_main'].unique())

ct_str = 'T cells'
combined = combined_all[combined_all.obs['given_cell_type_main']==ct_str]

In [ ]:
n_cell_per_donor = combined.obs['batch_sample_id'].value_counts().sort_values(ascending=True)

In [ ]:
# could set min_cell_counts based on quantile:
# min_cell_counts = n_cell_per_donor.quantile(0.25)//2
# print(min_cell_counts)

min_cell_counts = 500

print('num partitions per donor:')
n_part_df = (n_cell_per_donor[n_cell_per_donor>=min_cell_counts]//min_cell_counts).value_counts().sort_index()
print(n_part_df)
print('tot cell partitions:',sum([n_part_df.index[i] * n_part_df.iloc[i] for i in range(n_part_df.shape[0])]))

In [ ]:
combined.X[:10,:10]

In [ ]:
# $EPHEMERAL/pyspoc_Terekhova_data

In [ ]:
# cells per bucket, top genes, genes per gene-bucket

OUT = Path(os.environ["EPHEMERAL"]) / "pyspoc_Terekhova_data" 
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
combined.obs.shape

In [ ]:
def get_subsample(ids, n_sample, mode="partition", seed=None):
    """
    mode="partition": disjoint blocks from one shuffle
    """
    ids = pd.Series(ids)
    if len(ids) < n_sample:
        print(f'{len(ids)} < {n_sample}')
        return []
            
    if mode == "partition":
        shuffled = ids.sample(frac=1, random_state=seed).tolist()
        n_parts = len(shuffled) // n_sample
        return [shuffled[r * n_sample:(r + 1) * n_sample] for r in range(n_parts)]
    
    raise ValueError(f"unknown  mode {mode}")

In [ ]:
# get_subsample([1,2,3,4,5],5)

In [ ]:
import zlib

In [ ]:
min_tot_site=500
ct_str_short = 'T'
ct_str = 'T cells'

combined = sc.read_h5ad(OUT / f'combined_gex_terekhova_minThresh{min_tot_site}_ct{ct_str_short}.h5ad')

In [ ]:
combined.shape

In [ ]:
# ct_str_short = 'T'
# ct_str = 'T cells'
# combined = combined_all[combined_all.obs['given_cell_type_main']==ct_str]
# combined.write_h5ad(OUT / f'combined_gex_terekhova_minThresh{min_tot_site}_ct{ct_str_short}.h5ad')

In [ ]:
combined.shape

In [ ]:
n_cell_per_donor = combined.obs['batch_sample_id'].value_counts().sort_values(ascending=True)

In [ ]:
n_cell_per_donor.quantile(0.1)

In [ ]:
hvg = {}
n_max = 2000
for s in combined.obs["batch_sample_id"].unique():
    res = sc.pp.highly_variable_genes(combined[combined.obs["batch_sample_id"] == s],
                                      n_top_genes=n_max, inplace=False)
    hvg[s] = res[res["highly_variable"]].sort_values("dispersions_norm", ascending=False).copy()

In [ ]:
import gseapy as gp
hallmark = gp.get_library("MSigDB_Hallmark_2020", organism="Human")

In [ ]:
# h5_gene_int = set(combined.var_names).intersection(*[set(a.var_names) for a in h5_objs])

In [ ]:
min_frac = 3
key = "batch_sample_id"

ok = np.ones(combined.n_vars, dtype=bool)
for b in combined.obs[key].unique():
    X = combined[combined.obs[key] == b].X
    frac = np.asarray((X > 0).sum(axis=0)).ravel()
    ok &= frac >= min_frac

common_genes = combined.var_names[ok]
print(len(common_genes), "genes well expressed in every batch")

In [ ]:
hallmark_aging_names = {
    "ir" : "Inflammatory Response",
    "oxphos" : "Oxidative Phosphorylation",
    "p53" : "p53 Pathway"
}

hallmark_genes = {}

# print lens / len of union
v = set()
for kv in hallmark_aging_names.keys():
    full_v = hallmark_aging_names[kv]
    v1 = hallmark[full_v]
    print(f'individual set: {len(v1)}')
    
    int_with_h5 = set(v1).intersection(common_genes)
    print(f'int with h5: ', len(int_with_h5))
    v = v|set(int_with_h5)
    
    hallmark_genes[kv] = int_with_h5
    
print(f'union: {len(v)}')

In [ ]:
custom_genes = {}
min_cells_across_sample=0.01

for kv in hallmark_genes.keys():
    genes = list(hallmark_genes[kv])

    custom_genes[kv] = genes
    print(kv,len(custom_genes[kv]))
    custom_genes[kv].sort()

In [ ]:
# IF K IS SET TO 0 - PASS CUSTOM GENE LIST
TOT_HVG_LEN = 0
CELL_BUCKET_LEN = 1000
GENE_BUCKET_LEN = 1000

if TOT_HVG_LEN == 0:
    
    custom_gene_id = 'oxphos'
    custom_gene_sufx = f'MSigDb_{custom_gene_id}'
    custom_gene_lst = custom_genes[custom_gene_id]
    custom_gene_meta_str = f'using MSigDb 2020;{hallmark_aging_names[custom_gene_id]};min_pct_cell_with_gene_exp={min_cells_across_sample}'
    
    N = CELL_BUCKET_LEN
    P = len(custom_gene_lst)
    
    save_folder_str = f"customGenes_{custom_gene_id}"

else:
    
    N = CELL_BUCKET_LEN
    P = GENE_BUCKET_LEN
    K = TOT_HVG_LEN
    
    # OTHERWISE: EITHER TAKE TOP K GENES IF K = P
    # OR PARTITION GENE SETS
    do_topK = TOT_HVG_LEN == P
    
    if do_topK:
        save_folder_str = "topK"
    else:
        save_folder_str = f"geneSub_K{K}"

    custom_gene_meta_str = ""
    
save_folder = f"terekhova_N{N}_P{P}_{save_folder_str}"
meta_arr_folder = f"extraInfo_{save_folder}"
print(save_folder)

(OUT / save_folder).mkdir(exist_ok=True)

rows = []

# # SAVE LOOP
for s in n_cell_per_donor[n_cell_per_donor>=N].index:
    
    # PARTITION CELLS
    sub = combined[combined.obs["batch_sample_id"] == s]    
    cell_seed = np.array([0, zlib.crc32(f"{s}_cells".encode())]) 
    cell_subs = get_subsample(sub.obs.index,N,seed=cell_seed)
    
    gene_parts = None
    gene_seed= None
    
    common_sufx = f"ct{ct_str_short}_N{N}_P{P}"
    
    # FIND GENE SETS
    if TOT_HVG_LEN == 0: 
        #CUSTOM AT K=0
        gene_parts = [custom_gene_lst]
        common_sufx +='_' + custom_gene_sufx

    else:
        #HIGHLY VAR / TOP K OR PARTITION TOP K TO BUCKETS OF SIZE P
        hvg[s] = hvg[s].sort_values(by='dispersions_norm',ascending=False)    
        genes = hvg[s][:TOT_HVG_LEN].index

        if P == TOT_HVG_LEN:
            gene_parts = [genes]
        else:
            gene_seed = np.array([0, zlib.crc32(f"{s}_{save_folder_str}".encode())]) 
            gene_parts = get_subsample(genes,P,seed=gene_seed)    
    
    seed_str = f"{cell_seed[0]}:{cell_seed[1]}"
    gene_seed_str = "" if gene_seed is None else f"{gene_seed[0]}:{gene_seed[1]}"
    
    # SAVING LOOP
    for b, cb in enumerate(cell_subs):
        
        for g, gb in enumerate(gene_parts):
            
            #SUBSET H5
            X_h5 = sub[cb, gb]
                      
            #SET DATASET FILE NAME              
            name = (f"{s}_cells{b}_{common_sufx}" if (do_topK | K==0)
                    else f"{s}_cells{b}_geneid{g}_{common_sufx}")
            
            path = OUT / save_folder / f"{name}.npy"
            # print(path)

            #SAVE AS NPY
            X = X_h5.X.toarray() if hasattr(X_h5.X, "toarray") else np.asarray(X_h5.X)
            np.save(path, X.astype(np.float32))

            assert X.shape[0] == N
            assert X.shape[1] == P
            
            #SAVE METADATA
            rows.append({
                "safe_data_id": name,
                "sample_id": s,"path": path,
                "N": N,"P": len(gb),"mode": save_folder,
                "cell_sample_index": b,
                "tot_cell_sample": len(cell_subs),"cell_seed": seed_str,
                "gene_seed": gene_seed_str,
                "gene_sample_index": g,"tot_gene_sample": len(gene_parts),
                "TOT_HVG_LEN": TOT_HVG_LEN,"cell_type" : ct_str,
                "batch_donor_id" : X_h5.obs['batch_donor_id'].unique()[0],
                "age" : X_h5.obs['age'].unique()[0],
                "custom_gene_meta_str" : custom_gene_meta_str
            })
    #     break
    # break

In [ ]:
# X_h5.obs

In [ ]:
rows_df = pd.DataFrame(rows)
manifest = OUT / f"manifest_{save_folder}.csv"
rows_df.to_csv(manifest, index=False)

homepath = Path('~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/metadata_files')
rows_df.to_csv(homepath / f"{save_folder}.csv", index=False)